In [128]:
!pip install fuzzywuzzy

In [129]:
import os
import re
from fuzzywuzzy import fuzz

import nltk
nltk.download('punkt')
# nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
def preprocess_text(text):
    tokens = word_tokenize(text)

    return tokens

def preprocess_data(data_dir):
    data = []
    word_to_idx = {'<pad>': 0, '<unk>': 1}
    idx = 2

    for file_name in os.listdir(data_dir):
        if file_name.endswith('.ann'):
            text_file = os.path.join(data_dir, file_name.replace('.ann', '.txt'))
            with open(os.path.join(data_dir, file_name), 'r') as ann_file, open(text_file, 'r') as text_file:
                text = text_file.read()
                text = preprocess_text(text)
                all_keywords={}
                keywords_class = {}
                relations = []
                for line in ann_file:

                    line = line.strip()
                    if line.startswith('T'):
                        # _, label, start, end, phrase = line.split('\t')
                        id, labels, phrase = line.strip().split('\t')
                        label, start, end = labels.split()
                        start, end = int(start), int(end)
                        phrase_tokens = preprocess_text(phrase)
                        for token in phrase_tokens:
                            if token not in word_to_idx:
                                word_to_idx[token] = idx
                                idx += 1
                        if phrase not in all_keywords:
                            all_keywords[int(id[1:])] = phrase_tokens

                        if phrase not in all_keywords:
                            keywords_class[int(id[1:])] = label
                    elif line.startswith('R'):
                        relation_type, arg1, arg2 = re.split(r'\s+', line)[1:]
                        arg1 = int(arg1.split(':')[1][1:])
                        arg2 = int(arg2.split(':')[1][1:])
                        relations.append((arg1, arg2, relation_type))

                    elif line.startswith('*'):
                        # print(line)
                        relation_type, args = re.split(r'\s+', line, 2)[1:]
                        n_args = len(args.split())
                        args = args.split()
                        for i in range(n_args):
                            for j in range(i+1,n_args):
                                # print(args[i], args[j])
                                relations.append((int(args[i][1:]), int(args[j][1:]), relation_type))

                for i in all_keywords:
                    for j in all_keywords:
                        if j <= i:
                            continue
                        if (i,j,'Hyponym-of' not in relations) or (i,j,'Synonym-of' not in relations):
                            if keywords_class[i] == keywords_class[j]:
                                relations.append((i,j,'None'))

                for arg1, arg2, relation_type in relations:
                    phrase1 = all_keywords[arg1]
                    phrase2 = all_keywords[arg2]
                    relation_id = 0 if relation_type == 'Hyponym-of' else 1 if relation_type == 'Synonym-of' else 2
                    data.append({'text': text, 'phrase1': phrase1, 'id1': arg1, 'phrase2': phrase2, 'id2': arg2, 'relation': relation_id})
    return data, word_to_idx



def compare_strings(l1, l2):
    s1 = ""
    s2 = ""
    if (l1 == l2):
        # print(l1, l2)
        return False, (l1, l2)
    if (len(l1) == 1) and (len(l2) == 1):
        s1 = l1[0]
        s2 = l2[0]
    elif len(l1) == 1:
        s1 = l1[0].lower().strip("()")
        for i in range(len(l2)):
            if l2[i] == l2[i].split("-"):
                s2 += l2[i][0]
            else:
                hyphensplit = l2[i].split("-")
                for word in hyphensplit:
                    if word:
                        s2 += word[0].lower()


    elif len(l2) == 1:
        s2 = l2[0].lower().strip("()")
        for i in range(len(l1)):
            if l1[i] == l1[i].split("-"):
                s1 += l1[i][0]
            else:
                hyphensplit = l1[i].split("-")
                for word in hyphensplit:
                    if word:
                        s1 += word[0].lower()
    else:
        return False, ("blah", "blah")

    similarity_score = fuzz.ratio(s1, s2)
    # print(s1, s2)
    return (similarity_score > 80), (s1, s2)


In [131]:
df_tokenized = pd.read_pickle('task3df_dev.pkl')

## Classifying Synonymy Relations

In [ ]:
filtered_df = df_tokenized


tp, fp, tn, fn= 0, 0, 0, 0
for index, row in filtered_df.iterrows():
    # print(row['phrase1'], row['phrase2'])
    t_value, (s1, s2) = compare_strings(row['phrase1'], row['phrase2'])
    if (t_value):
        if (row['relation'] == 1):
            tp += 1
        else:
            # print(row['phrase1'], row['phrase2'])
            # print(s1, s2)
            # print("wrong")
            fp += 1
    else:
        if (row['relation'] != 1):
            tn += 1
        else:
            fn += 1

        # print(row['phrase1'], row['phrase2'])
    # print(row['phrase1'], row['phrase2'])
# print(filtered_df[['phrase1', 'phrase2']])

print(tp, tn, fp, fn)

In [134]:
print(tp/(tp+fn))
print(tp/(tp+fp))

0.4222222222222222
0.1532258064516129
